# Setup

In [ ]:
import pandas as pd
import json
import requests
import numpy as np
from bs4 import BeautifulSoup
from bs4 import XMLParsedAsHTMLWarning
import warnings
warnings.filterwarnings("ignore", category=XMLParsedAsHTMLWarning)
import re
import io, os
import csv
import boto3
from botocore.exceptions import ClientError
import openai
from openai import OpenAI
import base64
from PIL import Image
import anthropic
from anthropic import Anthropic

In [ ]:
data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover'
ctd_data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/CTD_data'
pubtator_data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/pubtator_data/pdfs'
input_data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockInput'
output_data_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockOutput'
figures_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockFigures'
supplement_path = '/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockSupplement'

In [ ]:
pm_pmc_dict = {
    '9118903': '1469763',
    '15154616': '2275409',
    '15811834': '1278483',
    '15953866': '2782200',
    '16393664': '1332662',
    '17118201': '1676006',
    '17183729': '1762369',
    '17524151': '1890552',
    '18280460': '2279149',
    '18299717': '2243244',
    '19114014': '2631511',
    '19617202': '3136680',
    '19654922': '2717139',
    '20801540': '2987498',
    '21294877': '3042928',
    '22110480': '3205718',
    '22112310': '3279445',
    '22228119': '3349910',
    '22562770': '3440069',
    '22571318': '3485091',
    '22747577': '3489685',
    '22808131': '3392256',
    '23006841': '3488501',
    '23349674': '3551951',
    '23568217': '3620667',
    '24039778': '3769399',
    '24403256': '3892387',
    '24413757': '3907846',
    '25361045': '4245613',
    '25859307': '4387225',
    '26039251': '4454644',
    '26492236': '4632778',
    '26714306': '4695093',
    '27153807': '4859981',
    '27397436': '5298042',
    '27517156': '5295416',
    '27736846': '5129817',
    '28637469': '5480112',
    '29045497': '5646803',
    '29380240': '6060758',
    '30104959': '6077207',
    '30114287': '6095593',
    '31064381': '6503545',
    '31378678': '6706627',
    '32298299': '7161975',
    '34908893': '8640676',
    '36638633': '10016736',
    '37047793': '10095583',
    '25100957': '4104784',
    '19889203': '3003513'
}

pmids = list(pm_pmc_dict.keys())
pmcids = list(pm_pmc_dict.values())

len(pmcids)

In [ ]:
import os, time, requests
import xml.etree.ElementTree as ET
from urllib.parse import urlparse

os.makedirs(pubtator_data_path, exist_ok=True)
BASE_URL = "https://pmc-oa-opendata.s3.amazonaws.com"
session = requests.Session()
downloaded, existing, failed = [], [], []

def is_valid_pdf(path):
    if not os.path.isfile(path): return False
    with open(path, "rb") as f: return f.read(5) == b"%PDF-"

def to_https(url):
    if not url.startswith("s3://"): return url
    p = urlparse(url)
    return f"https://{p.netloc}.s3.amazonaws.com/{p.path.lstrip('/')}"

def download_pdf(pmcid, output_path):
    r = session.get(BASE_URL, params={
        "list-type": "2", "prefix": f"{pmcid}.", "delimiter": "/"
    }, timeout=30)
    r.raise_for_status()

    root = ET.fromstring(r.content)
    ns = {"s3": "http://s3.amazonaws.com/doc/2006-03-01/"}
    versions = [x.text.rstrip("/") for x in root.findall(
        ".//s3:CommonPrefixes/s3:Prefix", ns) if x.text]

    if not versions: return False, "Article not found"

    candidates = []
    for version in versions:
        r = session.get(f"{BASE_URL}/{version}/{version}.json", timeout=30)
        if r.status_code == 404: continue
        r.raise_for_status()
        meta = r.json()
        if meta.get("pdf_url"):
            candidates.append((
                str(meta.get("is_manuscript", "no")).lower() == "yes",
                version, to_https(meta["pdf_url"])
            ))

    if not candidates: return False, "No PDF available"

    errors = []
    for _, version, url in sorted(candidates):
        try:
            with session.get(url, stream=True, timeout=120) as r:
                r.raise_for_status()
                with open(output_path, "wb") as f:
                    for chunk in r.iter_content(1024 * 1024):
                        if chunk: f.write(chunk)
            if is_valid_pdf(output_path): return True, version
            errors.append(f"{version}: Invalid PDF")
        except requests.RequestException as e:
            errors.append(f"{version}: {e}")
        if os.path.exists(output_path): os.remove(output_path)

    return False, "; ".join(errors)

for pmid, pmcid in pm_pmc_dict.items():
    pmc = f"PMC{str(pmcid).removeprefix('PMC')}"
    path = os.path.join(pubtator_data_path, f"{pmc}.pdf")

    if is_valid_pdf(path):
        existing.append(pmc)
        print(f"Already exists: {pmc}")
        continue

    try:
        success, msg = download_pdf(pmc, path)
        if success:
            downloaded.append(pmc)
            print(f"Saved PDF: {pmc}")
        else:
            failed.append((pmid, pmc, msg))
            print(f"⚠️ {pmc}: {msg}")
    except Exception as e:
        failed.append((pmid, pmc, str(e)))
        print(f"⚠️ {pmc}: {e}")
    time.sleep(0.5)

print(f"\nExisting: {len(existing)} | Downloaded: {len(downloaded)} | Failed: {len(failed)}")
for pmid, pmc, reason in failed:
    print(f"{pmc} (PMID {pmid}): {reason}")

# Run LLM

In [ ]:
def createBaselinePromptPDF():

    systemPrompt = """You are an AI assistant that extracts relationships between chemicals and genes from scientific text."""

    mainPrompt = f"""Extract all relationships between chemicals and genes from the text below. Format the output as CSV with one relationship per row.

Columns: "explanation", "chemicalName", "relationship", "geneName", "source"

The relationship must follow this format: "increase <type>", "decrease <type>", or "affect <type>" 
Choose from the following relationship types:
cotreatment, binding, reaction, activity, localization, expression, abundance, mutagenesis, response to chemical, stability, splicing, folding, transport, uptake, import, secretion, export, metabolic processing, chemical synthesis, degradation, acetylation, acylation, alkylation, amination, carbamoylation, carboxylation, cleavage, ethylation, glycation, glycosylation, N-linked glycosylation, O-linked glycosylation, glucuronidation, hydrolysis, hydroxylation, lipidation, geranoylation, farnesylation, myristoylation, palmitoylation, prenylation, methylation, nitrosation, nucleotidylation, oxidation, phosphorylation, sulfation, reduction, ribosylation, ADP-ribosylation, ubiquitination, sumoylation, glutathionylation, polymerization

Output only the CSV with no additional commentary.
"""

    return systemPrompt, mainPrompt

In [ ]:
rows = []
for pmid, pmcid in pm_pmc_dict.items():
    pmc_full = f"PMC{pmcid}"
    pdf_path = os.path.join(pubtator_data_path, f"{pmc_full}.pdf")
    
    if os.path.exists(pdf_path):
        with open(pdf_path, "rb") as f:
            system_prompt, main_prompt = createBaselinePromptPDF()
        
        rows.append({
            'pmid': pmid,
            'path': pdf_path,
            'system_prompt': system_prompt,
            'main_prompt': main_prompt
        })
    else:
        print(f"⚠️ PDF not found for PMID {pmid} ({pmc_full})")

pdf_df = pd.DataFrame(rows)
print(f"Found {len(pdf_df)} PDFs")

pdf_df['gpt4.1_output'] = ''
pdf_df

In [ ]:
gpt_key = ''

openai_client = OpenAI(api_key=gpt_key)


def call_gpt(system_prompt, main_prompt, pdf_path=None, model="gpt-4.1"):
    content = []

    if pdf_path and os.path.exists(pdf_path):
        with open(pdf_path, "rb") as f:
            pdf_data = base64.standard_b64encode(f.read()).decode("utf-8")
        content.append({
            "type": "input_file",
            "filename": os.path.basename(pdf_path),
            "file_data": f"data:application/pdf;base64,{pdf_data}"
        })

    content.append({
        "type": "input_text",
        "text": main_prompt
    })

    response = openai_client.responses.create(
        model=model,
        instructions=system_prompt,
        input=[{
            "role": "user",
            "content": content
        }],
        max_output_tokens=10000
    )
    return response.output_text

In [ ]:
def run_gpt(df):
    for idx, row in df.iterrows():
        if pd.notna(row.get("gpt4.1_output")):
            existing = row.get("gpt4.1_output")
            if pd.notna(existing) and str(existing).strip() != "":
                continue  # Skip already processed rows
        
        try:
            response = call_gpt(row["system_prompt"], row["main_prompt"], pdf_path=row.get("path"))
            df.at[idx, "gpt4.1_output"] = response
            print(f"done with row {idx}")
        except Exception as e:
            df.at[idx, "gpt4.1_output"] = f"ERROR: {str(e)}"
    
    return df

In [ ]:
# output_df = run_gpt(pdf_df)
# output_df.to_csv('/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockOutput/baseline_output.csv')

# Process output

In [ ]:
def extract_csv(df, source_col="gpt4.1_output"):
    def extract_block(text):
        if pd.isna(text):
            return pd.NA
        
        # Match ```csv ... ``` OR ``` ... ```
        pattern = r"```(?:csv)?\s*(.*?)```"
        match = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
        
        if match:
            return match.group(1).strip()
        
        return pd.NA  # No CSV block found

    df["gpt_output"] = df[source_col].apply(extract_block)
    return df

output_df = pd.read_csv(f'{output_data_path}/baseline_output.csv').drop(['Unnamed: 0'], axis=1)
df = extract_csv(output_df)
df

In [ ]:

import pandas as pd
import io
import csv

# Reload original data (important!)
raw_df = pd.read_csv("/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockOutput/baseline_output.csv")

def repair_csv(text):
    text = str(text).strip()

    # Extract last Markdown code block, if present
    if "```" in text:
        blocks = text.split("```")
        if len(blocks) >= 3:
            text = blocks[-2].removeprefix("csv").strip()

    lines = text.splitlines()
    if not lines:
        return ""

    header = next(csv.reader([lines[0]]))
    expected = len(header)
    names = [c.strip().lower() for c in header]
    merge_idx = next(
        (names.index(c) for c in
         ["genename", "explanation", "reasoning", "description"]
         if c in names), 0
    )

    output = io.StringIO()
    writer = csv.writer(output)
    writer.writerow(header)

    for line in lines[1:]:
        if not line.strip():
            continue
        try:
            parts = next(csv.reader([line]))
        except csv.Error:
            parts = line.split(",")

        if len(parts) > expected:
            extra = len(parts) - expected
            i = merge_idx
            parts = parts[:i] + [",".join(parts[i:i+extra+1])] + parts[i+extra+1:]

        writer.writerow(parts)

    return output.getvalue()


def extract_data(df):
    required = ["explanation", "chemicalName", "relationship",
                "geneName", "source"]
    extracted = []

    for idx, row in df.iterrows():
        try:
            text = row["gpt4.1_output"]
            if pd.isna(text) or not str(text).strip():
                continue

            text = repair_csv(text)
            temp = pd.read_csv(
                io.StringIO(text),
                engine="python",
                skipinitialspace=True,
                on_bad_lines="warn"
            )

            temp.columns = temp.columns.str.strip()

            for col in required:
                if col not in temp.columns:
                    temp[col] = pd.NA

            temp = temp[required].copy()
            temp.insert(0, "pmid", row["pmid"])
            extracted.append(temp)

        except Exception as e:
            print(f"Error at PMID {row['pmid']}: {e}")

    return (pd.concat(extracted, ignore_index=True)
            if extracted else pd.DataFrame(columns=["pmid"] + required))


In [ ]:
from __future__ import annotations
from pathlib import Path
import pandas as pd, numpy as np, io, csv, re, os, time
from typing import Optional
from IPython.display import display

# Set this path to your existing CTD_data folder.
CTD_DIR = Path("/Users/sc/nlp_4_ctd_code/data_json_clover/CTD_data")
BASELINE_CSV = Path("/Users/sc/nlp_4_ctd_code/data_json_clover/BedrockOutput/baseline_output.csv")
SAVE_DIR = Path("baseline_ctd_matching")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
START = time.perf_counter()
def log(msg): print(f"[{time.perf_counter()-START:.1f}s] {msg}")

ctd_chemicals = pd.read_csv(f"{ctd_data_path}/CTD_chemicals.csv", skiprows = 27)
ctd_genes = pd.read_csv(f"{ctd_data_path}/CTD_genes.csv", skiprows = 27)

ctd_ixns = pd.read_csv(f"{ctd_data_path}/CTD_chem_gene_ixns.csv", skiprows = 27).drop(labels=0)
ctd_ixns["InteractionActionsList"] = ctd_ixns["InteractionActions"].apply(lambda x: [item.replace("^", " ") for item in x.split("|")])
ctd_ixns["pmcidsList"] = ctd_ixns["PubMedIDs"].apply(lambda x: [str(item) for item in x.split("|")])
ctd_ixns["GeneID"] = ctd_ixns["GeneID"].apply(int)
ctd_ixns["GeneID"] = ctd_ixns["GeneID"].apply(str)
ctd_ixns["ChemicalID"] = ctd_ixns["ChemicalID"].apply(str)


In [ ]:
raw_df = pd.read_csv(BASELINE_CSV)
print("Input columns:", raw_df.columns.tolist())

def extract_baseline(df):
    required = ["explanation", "chemicalName", "relationship", "geneName", "source"]
    results = []
    for _, row in df.iterrows():
        value = row.get("gpt4.1_output")
        if pd.isna(value) or not str(value).strip(): continue
        try:
            parsed = pd.read_csv(io.StringIO(repair_csv(str(value))), engine="python", on_bad_lines="warn")
            parsed.columns = parsed.columns.str.strip()
            for col in required:
                if col not in parsed: parsed[col] = pd.NA
            parsed = parsed[required].copy()
            parsed.insert(0, "pmid", row["pmid"])
            results.append(parsed)
        except Exception as exc:
            print(f"PMID {row['pmid']}: {exc}")
    if not results: raise ValueError("No baseline output rows parsed; check BASELINE_CSV and gpt4.1_output.")
    return pd.concat(results, ignore_index=True)

extracted_df = extract_baseline(raw_df)
print("Extracted:", extracted_df.shape)
display(extracted_df.head())


In [ ]:
baseline_df = extracted_df.copy()
for col in ["chemicalID", "geneID"]: baseline_df[col] = pd.NA
baseline_df["geneSymbol"] = baseline_df["geneName"]
# Keep section/source as in original extraction; remaining optional fields are handled downstream.
ctd_chemicals = pd.read_csv(CTD_DIR / "CTD_chemicals.csv", skiprows=27)
ctd_genes = pd.read_csv(CTD_DIR / "CTD_genes.csv", skiprows=27)

def build_ctd_chemical_lookup(ctd_chemicals_df):
    chem_id_to_info = {}
    name_to_chem_id = {}

    # Make column names consistent
    ctd_chemicals_df = ctd_chemicals_df.rename(columns=lambda x: x.strip())

    for _, row in ctd_chemicals_df.iterrows():
        chem_name = str(row.get("# ChemicalName", "") or row.get("Chemical Name", "")).strip()
        chem_id = str(row.get("ChemicalID", "") or row.get("Chemical ID", "")).replace("MESH:", "").upper()
        
        # synonyms
        mesh_syn = str(row.get("MESHSynonyms", "")).split("|") if row.get("MESHSynonyms") else []
        ctd_syn = str(row.get("CTDCuratedSynonyms", "")).split("|") if row.get("CTDCuratedSynonyms") else []
        synonyms = [s.strip() for s in mesh_syn + ctd_syn if s.strip()]

        chem_id_to_info[chem_id] = {"chem_name": chem_name, "synonyms": synonyms}

        if chem_name:
            name_to_chem_id[chem_name.upper()] = chem_id

        for syn in synonyms:
            name_to_chem_id[syn.upper()] = chem_id

    return chem_id_to_info, name_to_chem_id



def normalize_gene_id(x):
    """Convert GeneID to clean string or None"""
    if pd.isna(x):
        return None
    try:
        return str(int(float(x)))
    except (ValueError, TypeError):
        return str(x).strip()

def build_ctd_gene_lookups(ctd_genes_df):
    """
    Build lookup dictionaries from CTD genes DataFrame.
    
    Returns:
      geneid_to_name:   {GeneID -> canonical GeneName}
      geneid_to_symbol:{GeneID -> GeneSymbol}
      name_to_geneid:  {UPPERCASE name/symbol/synonym -> GeneID}
      alt_to_geneid:   {AltGeneID -> GeneID}
    """
    geneid_to_name = {}
    geneid_to_symbol = {}
    name_to_geneid = {}
    alt_to_geneid = {}

    for _, row in ctd_genes_df.iterrows():
        gene_id = normalize_gene_id(row.get("GeneID"))
        if not gene_id:
            continue

        symbol = row.get("# GeneSymbol")
        # print(symbol)
        name = row.get("GeneName")
        synonyms = row.get("Synonyms")
        alt_ids = row.get("AltGeneIDs")

        # Canonical name: prefer GeneName over symbol
        canonical_name = name if isinstance(name, str) and name.strip() else symbol
        if not isinstance(canonical_name, str) or not canonical_name.strip():
            continue

        geneid_to_name[gene_id] = canonical_name
        geneid_to_symbol[gene_id] = symbol.strip() if isinstance(symbol, str) else None

        # Index symbol + name
        for n in [symbol, name]:
            if isinstance(n, str) and n.strip():
                name_to_geneid[n.upper()] = gene_id

        # Index synonyms
        if isinstance(synonyms, str) and synonyms.strip():
            for syn in synonyms.split("|"):
                syn = syn.strip()
                if syn:
                    name_to_geneid[syn.upper()] = gene_id

        # Index alternate GeneIDs
        if isinstance(alt_ids, str) and alt_ids.strip():
            for alt in alt_ids.split("|"):
                alt = normalize_gene_id(alt)
                if alt:
                    alt_to_geneid[alt] = gene_id

    return geneid_to_name, geneid_to_symbol, name_to_geneid, alt_to_geneid


def update_chemical(df, chem_id_to_info, name_to_chem_id, verbose=True):
    """
    Update chemicalName and chemicalID efficiently using prebuilt lookup tables.
    Flags:
        chemicalFlag = 0 : found normally
        chemicalFlag = 1 : not found
    """
    df = df.copy()
    df['chemicalFlag'] = 0

    # Normalize chemicalID column
    df['chemicalID'] = df['chemicalID'].apply(
        lambda x: str(x).upper().replace("MESH:", "").strip() if pd.notna(x) else None
    )
    df['chemicalName'] = df['chemicalName'].fillna("").astype(str)

    # Precompute normalized name_to_chem_id dict
    norm_name_to_id = {
        k.upper().replace("-", "").replace(" ", ""): v
        for k, v in name_to_chem_id.items()
    }

    for i, row in df.iterrows():
        chem_orig = row['chemicalName']
        chemicalName = chem_orig.strip()
        chemicalID = row['chemicalID']
        chemical_found = False

        # 1️⃣ Exact match by chemicalID
        if chemicalID and chemicalID in chem_id_to_info:
            df.at[i, 'chemicalName'] = chem_id_to_info[chemicalID]['chem_name']
            if verbose:
                print(f"Row {i} - Exact ID match: {chem_id_to_info[chemicalID]['chem_name']}")
            chemical_found = True
            continue  # already found

        # 2️⃣ Exact match by normalized chemicalName
        query_upper = chemicalName.upper().replace("-", "").replace(" ", "")
        if query_upper in norm_name_to_id:
            matched_id = norm_name_to_id[query_upper]
            df.at[i, 'chemicalID'] = matched_id
            df.at[i, 'chemicalName'] = chem_id_to_info[matched_id]['chem_name']
            if verbose:
                print(f"Row {i} - Exact name match: {chem_id_to_info[matched_id]['chem_name']}")
            chemical_found = True
            continue

        # 3️⃣ Partial match
        for key_norm, cid in norm_name_to_id.items():
            if query_upper and query_upper in key_norm:
                df.at[i, 'chemicalID'] = cid
                df.at[i, 'chemicalName'] = chem_id_to_info[cid]['chem_name']
                chemical_found = True
                if verbose:
                    print(f"Row {i} - Partial match: {chem_id_to_info[cid]['chem_name']}")
                break

        # 4️⃣ If still not found
        if not chemical_found:
            df.at[i, 'chemicalFlag'] = 1
            if not df.at[i, 'chemicalName']:
                df.at[i, 'chemicalName'] = chem_orig
            if verbose:
                print(f"Row {i} - Not found in CTD chemicals")
                print(f"  chemicalName: {chem_orig}")
                print(f"  chemicalID:   {chemicalID}")
                print("-" * 40)

    return df

def update_gene(df, geneid_to_name, geneid_to_symbol, name_to_geneid, alt_to_geneid, verbose=True):
    """
    Update geneName, geneID, and geneSymbol in the DataFrame using CTD gene lookups.

    Flags:
        geneFlag = 0 : found normally
        geneFlag = 1 : not found

    Additionally:
        oldGeneID = original geneID before normalization/update
    """
    df = df.copy()
    df["geneFlag"] = 0
    
    # ✅ Store original geneID before any modification
    df["oldGeneID"] = df["geneID"]
    df["oldGeneName"] = df["geneName"]
    df["oldGeneSymbol"] = df["geneSymbol"]

    for i, row in df.iterrows():
        original_geneID = row.get("geneID")
        geneID = normalize_gene_id(original_geneID)
        geneName = row.get("geneName")
        geneSymbol = row.get("geneSymbol", None)

        found = False

        # 1️⃣ Resolve by GeneID
        if geneID:
            if geneID in geneid_to_name:
                geneName = geneid_to_name[geneID]
                geneSymbol = geneid_to_symbol.get(geneID)
                found = True

            elif geneID in alt_to_geneid:
                parent_id = alt_to_geneid[geneID]
                geneID = parent_id
                geneName = geneid_to_name.get(parent_id)
                geneSymbol = geneid_to_symbol.get(parent_id)
                found = True
        # somewhere in here, geneSymbol is getting overwritten to geneName.
        
        # 2️⃣ Flags
        if not found:
            df.at[i, "geneFlag"] = 1
            if verbose:
                print(f"Row {i} not found in CTD genes")
                print(f"  geneName: {geneName}")
                print(f"  geneID:   {geneID}")
                print("-" * 40)
        else:
            df.at[i, "geneFlag"] = 0

        # ✅ Overwrite geneID (while oldGeneID is preserved)
        df.at[i, "geneName"] = geneName
        df.at[i, "geneID"] = geneID
        df.at[i, "geneSymbol"] = geneSymbol

    return df

In [ ]:
def update_relationship(df, verbose=True):
    # Allowed CTD relationships
    relationships = [
        "cotreatment", "binding", "reaction", "activity", "localization", "expression", "abundance", 
        "mutagenesis", "response to chemical", "stability", "splicing", "folding", "transport", 
        "uptake", "import", "secretion", "export", "metabolic processing", "chemical synthesis", 
        "degradation", "acetylation", "acylation", "alkylation", "amination", "carbamoylation", 
        "carboxylation", "cleavage", "ethylation", "glycation", "glycosylation", "N-linked glycosylation", 
        "O-linked glycosylation", "glucuronidation", "hydrolysis", "hydroxylation", "lipidation", 
        "geranoylation", "farnesylation", "myristoylation", "palmitoylation", "prenylation", 
        "methylation", "nitrosation", "nucleotidylation", "oxidation", "phosphorylation", "sulfation", 
        "reduction", "ribosylation", "ADP-ribosylation", "ubiquitination", "sumoylation", 
        "glutathionylation", "polymerization"
    ]
    lower_relationships = {r.lower() for r in relationships}

    # Prefix mapping: canonical form
    prefix_map = {
        'increase': 'increases', 'increases': 'increases', 'increased': 'increases', 'increasing': 'increases',
        'decrease': 'decreases', 'decreases': 'decreases', 'decreased': 'decreases', 'decreasing': 'decreases',
        'affect': 'affects', 'affects': 'affects', 'affected': 'affects', 'affecting': 'affects'
    }

    # Prefix exceptions: always "affects"
    affects_required = ['binding', 'cotreatment']

    # Chemical-chemical only relationships
    chem_chem_only = ['abundance', 'response to chemical', 'chemical synthesis']

    # Inhibitory terms replaced with "decrease"
    inhibit_terms = ['inhibit', 'inhibits', 'inhibited', 'inhibiting', 'inhibition']

    df = df.copy()
    df['relationshipWarningFlag'] = 0
    df['prefixMissingFlag'] = 0  # NEW: initialize flag

    if 'relationship' not in df.columns:
        df['relationship'] = ''

    # Normalize text: lowercase, remove quotes, strip spaces
    rel_series = df['relationship'].astype(str).str.lower().str.replace('"', '').str.strip()
    rel_series = rel_series.apply(lambda x: re.sub(r'\s+', ' ', x))

    # Replace inhibitory terms with "decrease"
    for term in inhibit_terms:
        rel_series = rel_series.str.replace(rf'\b{term}\b', 'decrease', regex=True)

    # Normalize prefix and apply rules
    def normalize(rel):
        words = rel.split()
        prefix = words[0] if words and words[0] in prefix_map else None
        core = ' '.join(words[1:]) if prefix else rel

        # NEW: set prefixMissingFlag if no prefix
        if not prefix:
            df.at[df.index[rel_series[rel_series == rel].index[0]], 'prefixMissingFlag'] = 1

        # Prefix exceptions
        if core in affects_required:
            return f"affects {core}"

        # Apply canonical prefix if prefix exists
        if prefix:
            new_prefix = prefix_map.get(prefix, prefix)
            return f"{new_prefix} {core}" if core else new_prefix

        return core

    rel_series = rel_series.apply(normalize)

    # Flag prefix and relationship validity (single pass)
    def flag_prefix(rel):

        words = rel.split()
        prefix = words[0] if words else None
        valid_prefixes = {'increase', 'increases', 'decrease', 'decreases', 'affect', 'affects'}
    
        # Flag if no valid prefix
        if prefix not in valid_prefixes:
            return 1
    
        core = ' '.join(words[1:])
    
        # chem-chem only relationships always invalid
        if core in chem_chem_only:
            return 1
    
        # Invalid if core not in allowed relationships
        return 0 if core in lower_relationships else 1

    df['relationship'] = rel_series
    df['relationshipWarningFlag'] = rel_series.apply(flag_prefix)

    if verbose:
        invalid_df = df[df['relationshipWarningFlag'] == 1]
        n_invalid = len(invalid_df)
        print(f"{n_invalid} invalid relationships detected.")
        if n_invalid > 0:
            cols_to_show = ['relationship', 'relationshipWarningFlag', 'prefixMissingFlag']
            for col in ['chemicalID', 'geneID', 'chemicalFlag', 'geneFlag']:
                if col in df.columns:
                    cols_to_show.insert(0, col)
            print(invalid_df[cols_to_show].head(20))

    return df


In [ ]:


# ---------------------------------------------------------------------------
# Small pure helpers
# ---------------------------------------------------------------------------

def _symbol_numeric_suffix(symbol) -> Optional[str]:
    """
    Extract the portion of a GeneSymbol starting from its first digit.
        FKBP1A -> "1A"
        IL6    -> "6"
        TP53   -> "53"
    Returns None if no digit is found or input is NaN.
    """
    if pd.isna(symbol):
        return None
    m = re.search(r"\d.*", str(symbol))
    return m.group(0) if m else None


def _tokenize(name) -> list:
    """Split a (already-cleaned) name string into lowercase tokens."""
    if pd.isna(name) or not str(name).strip():
        return []
    return str(name).strip().lower().split()


def _remove_parentheses(name) -> str:
    """
    Strip parenthesised substrings (including the parentheses themselves).
        "glyceraldehyde-3-phosphate dehydrogenase (GAPDH)"
        -> "glyceraldehyde-3-phosphate dehydrogenase"
    """
    if pd.isna(name):
        return ""
    return re.sub(r"\s*\(.*?\)", "", str(name)).strip()


def _suffix_in_tokens(suffix: Optional[str], tokens: list) -> bool:
    """
    Return True when *suffix* (e.g. "53") is represented in *tokens*.

    Single-token names  : suffix must appear *anywhere* in the token.
    Multi-token names   : at least one token must *end with* suffix.
    """
    if not suffix:
        return False
    suffix = suffix.lower()
    if len(tokens) == 1:
        return suffix in tokens[0]
    return any(tok.endswith(suffix) for tok in tokens)


def _normalize(s) -> str:
    """Lowercase, strip, coerce NaN -> empty string."""
    if pd.isna(s):
        return ""
    return str(s).strip().lower()


def _alt_count(alt_ids_str) -> int:
    """Count non-empty pipe-separated alternative gene IDs."""
    return len([a for a in str(alt_ids_str).split("|") if a.strip()])


def _symbol_is_missing(symbol) -> bool:
    """Return True if a geneSymbol value should be treated as absent."""
    if pd.isna(symbol):
        return True
    return str(symbol).strip() in ("", "-")


def _strip_dot_suffix(symbol) -> Optional[str]:
    """
    If a symbol contains a dot, return the portion before it.
        BRCA1.2  -> "BRCA1"
        TP53.1   -> "TP53"
        BRCA1    -> None  (no dot present)
    """
    if pd.isna(symbol):
        return None
    s = str(symbol).strip()
    if "." in s:
        return s.split(".")[0]
    return None


# ---------------------------------------------------------------------------
# CTD pre-processing  (call once, reuse across batches)
# ---------------------------------------------------------------------------

def prepare_ctd(ctd_genes: pd.DataFrame) -> dict:
    """
    Build fast lookup structures from the CTD gene table.

    Returns a dict with:
        "df"           : cleaned CTD dataframe with helper columns
        "id_index"     : {gene_id_str          -> list[row_index]}
        "symbol_index" : {normalised_symbol    -> list[row_index]}
        "name_index"   : {normalised_gene_name -> list[row_index]}
        "synonym_index": {normalised_synonym   -> list[row_index]}
    """
    ctd = ctd_genes.copy()
    ctd["GeneID"]       = ctd["GeneID"].astype(str)
    ctd["_symbol_norm"] = ctd["# GeneSymbol"].apply(_normalize)
    ctd["_name_norm"]   = ctd["GeneName"].apply(_normalize)
    ctd["_alt_count"]   = ctd["AltGeneIDs"].fillna("").apply(_alt_count)
    ctd["_syns"]        = ctd["Synonyms"].fillna("").str.lower().str.split("|")

    id_index: dict = {}
    for i, gid in ctd["GeneID"].items():
        if gid:
            id_index.setdefault(gid, []).append(i)

    symbol_index: dict = {}
    for i, sym in ctd["_symbol_norm"].items():
        if sym:
            symbol_index.setdefault(sym, []).append(i)

    name_index: dict = {}
    for i, name in ctd["_name_norm"].items():
        if name:
            name_index.setdefault(name, []).append(i)

    synonym_index: dict = {}
    for i, syns in ctd["_syns"].items():
        for s in syns:
            s = s.strip()
            if s:
                synonym_index.setdefault(s, []).append(i)

    return {
        "df": ctd,
        "id_index": id_index,
        "symbol_index": symbol_index,
        "name_index": name_index,
        "synonym_index": synonym_index,
    }


# ---------------------------------------------------------------------------
# Per-row resolution helpers
# ---------------------------------------------------------------------------

def _resolve_by_id(gene_id: str, ctd_lookup: dict) -> Optional[pd.Series]:
    """Priority 1: exact GeneID match."""
    rows = ctd_lookup["df"].loc[ctd_lookup["id_index"].get(gene_id, [])]
    return rows.iloc[0] if not rows.empty else None


def _resolve_by_symbol(symbol_norm: str, ctd_lookup: dict) -> Optional[pd.Series]:
    """Priority 2: exact normalised GeneSymbol match."""
    rows = ctd_lookup["df"].loc[ctd_lookup["symbol_index"].get(symbol_norm, [])]
    return rows.iloc[0] if not rows.empty else None


def _resolve_by_name(name_norm: str, name_tokens: list, ctd_lookup: dict) -> Optional[pd.Series]:
    """
    Priority 3: match via GeneName or Synonyms with alt_count tiebreaking.

    Cases
    -----
    1. Only GeneName hits  : prefer suffix-matching candidates; tiebreak by alt_count
    2. Both hits           : compare total alt_count of each group; pick winner's best
    3. Only Synonym hits   : pick highest alt_count
    """
    if not name_norm:
        return None

    ctd_df        = ctd_lookup["df"]
    name_index    = ctd_lookup["name_index"]
    synonym_index = ctd_lookup["synonym_index"]

    gn_rows  = ctd_df.loc[name_index.get(name_norm, [])]
    syn_rows = ctd_df.loc[synonym_index.get(name_norm, [])]

    # Deduplicate: remove synonym hits already present in gene-name hits
    if not gn_rows.empty and not syn_rows.empty:
        syn_rows = syn_rows[~syn_rows.index.isin(gn_rows.index)]

    # ---- Case 1: only GeneName hits ----
    if not gn_rows.empty and syn_rows.empty:
        suffix_matches = [
            r for _, r in gn_rows.iterrows()
            if _suffix_in_tokens(_symbol_numeric_suffix(r["# GeneSymbol"]), name_tokens)
        ]
        best_df = pd.DataFrame(suffix_matches) if suffix_matches else gn_rows
        return best_df.sort_values("_alt_count", ascending=False).iloc[0]

    # ---- Case 2: both hits ----
    if not gn_rows.empty and not syn_rows.empty:
        if gn_rows["_alt_count"].sum() >= syn_rows["_alt_count"].sum():
            return gn_rows.sort_values("_alt_count", ascending=False).iloc[0]
        else:
            return syn_rows.sort_values("_alt_count", ascending=False).iloc[0]

    # ---- Case 3: only Synonym hits ----
    if not syn_rows.empty:
        return syn_rows.sort_values("_alt_count", ascending=False).iloc[0]

    return None


# ---------------------------------------------------------------------------
# Dot-symbol resolution helper
# ---------------------------------------------------------------------------

def _apply_dot_check(best: pd.Series, ctd_lookup: dict) -> pd.Series:
    """
    If the resolved symbol contains a dot (e.g. BRCA1.2), strip the suffix
    and re-look up the base symbol in CTD to get the correct GeneID/GeneName.
    Returns the corrected row, or the original row with the dot stripped if
    no CTD entry is found for the base.
    """
    base = _strip_dot_suffix(best["# GeneSymbol"])
    if base is None:
        return best  # no dot, nothing to do

    hit = _resolve_by_symbol(_normalize(base), ctd_lookup)
    if hit is not None:
        return hit

    # No CTD entry for base symbol — strip the dot suffix in-place
    best = best.copy()
    best["# GeneSymbol"] = base
    return best


# ---------------------------------------------------------------------------
# Public entry point
# ---------------------------------------------------------------------------
def update_gene_symbol_(
    df: pd.DataFrame,
    ctd_genes: pd.DataFrame,
    ctd_lookup: Optional[dict] = None,
) -> pd.DataFrame:

    df = df.copy()

    df["oldGeneID"]     = df.get("geneID")
    df["oldGeneName"]   = df.get("geneName")
    df["oldGeneSymbol"] = df.get("geneSymbol")

    # ✅ Treat "-", " -", "- " etc. as missing across key gene columns
    for col in ["geneID", "geneSymbol", "geneName"]:
        if col in df.columns:
            df[col] = df[col].replace(r"^\s*-\s*$", "", regex=True).fillna("")

    if ctd_lookup is None:
        ctd_lookup = prepare_ctd(ctd_genes)

    # Vectorised pre-normalisation
    df["_name_norm"] = (
        df.get("geneName", pd.Series(dtype=str))
        .apply(_remove_parentheses)
        .apply(_normalize)
    )
    df["_symbol_norm"] = (
        df.get("geneSymbol", pd.Series(dtype=str))
        .apply(_normalize)
    )

    flags       = np.full(len(df), 2, dtype=int)
    new_symbols = df["geneSymbol"].copy() if "geneSymbol" in df.columns else pd.Series(index=df.index, dtype=str)
    new_ids     = df["geneID"].astype(str).copy()
    new_names   = df["geneName"].copy()   if "geneName"   in df.columns else pd.Series(index=df.index, dtype=str)

    for pos, (idx, row) in enumerate(df.iterrows()):
        raw_symbol  = row.get("geneSymbol")
        gene_id     = str(row.get("geneID", "")).strip()
        symbol_norm = row["_symbol_norm"]
        name_norm   = row["_name_norm"]
        name_tokens = _tokenize(name_norm)

        # Already valid symbol
        if not _symbol_is_missing(raw_symbol):
            if not gene_id:  # symbol present but ID missing — try to resolve ID from symbol
                best = _resolve_by_symbol(symbol_norm, ctd_lookup)
                if best is not None:
                    best = _apply_dot_check(best, ctd_lookup)
                    new_symbols.iloc[pos] = best["# GeneSymbol"]
                    new_ids.iloc[pos]     = best["GeneID"]
                    new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0
            continue
    
        # Priority 1: GeneID
        if gene_id:
            best = _resolve_by_id(gene_id, ctd_lookup)
            if best is not None:
                best = _apply_dot_check(best, ctd_lookup)
                new_symbols.iloc[pos] = best["# GeneSymbol"]
                new_ids.iloc[pos]     = best["GeneID"]
                new_names.iloc[pos]   = best["GeneName"]
                flags[pos] = 0
                continue

        # Priority 2: GeneSymbol
        if symbol_norm:
            best = _resolve_by_symbol(symbol_norm, ctd_lookup)
            if best is not None:
                best = _apply_dot_check(best, ctd_lookup)
                new_symbols.iloc[pos] = best["# GeneSymbol"]
                new_ids.iloc[pos]     = best["GeneID"]
                new_names.iloc[pos]   = best["GeneName"]
                flags[pos] = 0
                continue

        # Priority 3a: treat geneName as a symbol
        best = _resolve_by_symbol(name_norm, ctd_lookup)
        if best is not None:
            best = _apply_dot_check(best, ctd_lookup)
            new_symbols.iloc[pos] = best["# GeneSymbol"]
            new_ids.iloc[pos]     = best["GeneID"]
            new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0
            continue

        # Priority 3b: GeneName / Synonym
        best = _resolve_by_name(name_norm, name_tokens, ctd_lookup)
        if best is not None:
            best = _apply_dot_check(best, ctd_lookup)
            new_symbols.iloc[pos] = best["# GeneSymbol"]
            new_ids.iloc[pos]     = best["GeneID"]
            new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0

    df["geneSymbol"]     = new_symbols.values
    df["geneID"]         = new_ids.values
    df["geneName"]       = new_names.values
    df["geneSymbolFlag"] = flags

    df.drop(columns=["_name_norm", "_symbol_norm"], inplace=True)
    return df


In [ ]:
chem_id_to_info, name_to_chem_id = build_ctd_chemical_lookup(ctd_chemicals)
geneid_to_name, geneid_to_symbol, name_to_geneid, alt_to_geneid = build_ctd_gene_lookups(ctd_genes)

def refine(df, ctd_chemicals, ctd_genes):
    updated_df = update_chemical(
        df=df, 
        chem_id_to_info=chem_id_to_info, 
        name_to_chem_id=name_to_chem_id,
        verbose=False
    )

    updated_df = update_gene(
        updated_df,
        geneid_to_name,
        geneid_to_symbol,
        name_to_geneid,
        alt_to_geneid,
        verbose=False
    )
    updated_df = update_relationship(updated_df, verbose = False)
    return updated_df

refined_df = refine(baseline_df, ctd_chemicals, ctd_genes)
print("CTD refined:", refined_df.shape)
display(refined_df.head())
refined_df.to_csv(SAVE_DIR / "baseline_refined.csv", index=False)



# CLOVER baseline

In [ ]:
from pathlib import Path
import re
import pandas as pd
import numpy as np
import requests
from IPython.display import display

# Existing project layout from NLP4CTD_Baseline.ipynb and the combined notebook.
PROJECT_ROOT = Path('/Users/sc/nlp_4_ctd_code/data_json_clover')
if not PROJECT_ROOT.is_dir():
    PROJECT_ROOT = Path('/mnt/data') if Path('/mnt/data/baseline_refined.csv').is_file() else Path.cwd()

# Previous baseline notebook writes to baseline_ctd_matching/baseline_refined.csv.
# Also handle a copy next to this notebook, in BedrockOutput, or attached in /mnt/data.
INPUT_CANDIDATES = [
    PROJECT_ROOT / 'baseline_ctd_matching' / 'baseline_refined.csv',
    PROJECT_ROOT / 'BedrockOutput' / 'baseline_ctd_matching' / 'baseline_refined.csv',
    PROJECT_ROOT / 'BedrockOutput' / 'baseline_refined.csv',
    PROJECT_ROOT / 'baseline_refined.csv',
    Path.cwd() / 'baseline_ctd_matching' / 'baseline_refined.csv',
    Path.cwd() / 'baseline_refined.csv',
    Path('/mnt/data/baseline_refined.csv'),
]
BASELINE_CSV = next((p for p in INPUT_CANDIDATES if p.is_file()), None)
if BASELINE_CSV is None:
    raise FileNotFoundError('baseline_refined.csv not found. Checked:\n' + '\n'.join(map(str, INPUT_CANDIDATES)))

CTD_CANDIDATES = [
    PROJECT_ROOT / 'CTD_data',
    Path.cwd() / 'CTD_data',
    Path('/mnt/data/CTD_data'),
]
CTD_DIR = next((p for p in CTD_CANDIDATES if all((p / f).is_file() for f in
    ['CTD_chemicals.csv', 'CTD_genes.csv', 'CTD_chem_gene_ixns.csv'])), None)
if CTD_DIR is None:
    raise FileNotFoundError('Missing CTD_chemicals.csv / CTD_genes.csv / CTD_chem_gene_ixns.csv. Checked:\n' + '\n'.join(map(str, CTD_CANDIDATES)))

SAVE_DIR = PROJECT_ROOT / 'CLOVER_output_files' / 'baseline_ctd_matching'
SAVE_DIR.mkdir(parents=True, exist_ok=True)
print('Baseline input:', BASELINE_CSV)
print('CTD directory:', CTD_DIR)
print('Metrics output:', SAVE_DIR / 'baseline_precision_recall.csv')


In [ ]:
# Read the already-refined baseline; never overwrite the input DataFrame.
baseline = pd.read_csv(BASELINE_CSV, low_memory=False)
required = {'pmid', 'chemicalName', 'chemicalID', 'geneName', 'geneSymbol',
            'geneID', 'relationship', 'source', 'chemicalFlag',
            'relationshipWarningFlag', 'prefixMissingFlag'}
missing = required - set(baseline.columns)
if missing:
    raise ValueError(f'Input is not the expected baseline_refined.csv; missing columns: {sorted(missing)}')
baseline['pmid'] = pd.to_numeric(baseline['pmid'], errors='coerce').astype('Int64').astype('string')
baseline = baseline.dropna(subset=['pmid']).copy()
REFERENCE_PMIDS = set(baseline['pmid'])
print(f'Baseline predictions: {len(baseline):,}; PMIDs: {len(REFERENCE_PMIDS)}')
suspect_gene_fields = baseline['geneSymbol'].fillna('').astype(str).str.contains(',', regex=False).sum()
if suspect_gene_fields:
    print(f'WARNING: {suspect_gene_fields} geneSymbol values contain commas; inspect baseline_refined.csv for possible CSV field shifts before interpreting metrics.')

def read_ctd_csv(filename):
    path = CTD_DIR / filename
    return pd.read_csv(path, skiprows=27, low_memory=False).drop(index=0, errors='ignore').copy()

ctd_chemicals = read_ctd_csv('CTD_chemicals.csv')
ctd_genes = read_ctd_csv('CTD_genes.csv')
ctd_raw = read_ctd_csv('CTD_chem_gene_ixns.csv')
ctd_raw['GeneID'] = pd.to_numeric(ctd_raw['GeneID'], errors='coerce').astype('Int64').astype('string')
ctd_raw['ChemicalID'] = ctd_raw['ChemicalID'].astype('string')
ctd_raw['InteractionActionsList'] = ctd_raw['InteractionActions'].fillna('').apply(
    lambda s: [x.replace('^', ' ') for x in str(s).split('|') if x.strip()])
ctd_raw['pmidsList'] = ctd_raw['PubMedIDs'].fillna('').apply(
    lambda s: [x.strip() for x in str(s).split('|') if x.strip()])
ctd_raw['ref_pmids'] = ctd_raw['pmidsList'].apply(lambda ids: [x for x in ids if x in REFERENCE_PMIDS])
ctd_ixns = (ctd_raw[ctd_raw['ref_pmids'].apply(len) > 0]
            .explode('ref_pmids').rename(columns={'ref_pmids': 'pmid'}))
ctd_ixns = ctd_ixns[ctd_ixns['InteractionActionsList'].apply(len) == 1].copy()
ctd_ixns = ctd_ixns.dropna(subset=['ChemicalID', 'GeneID']).drop_duplicates(
    subset=['pmid', 'ChemicalID', 'GeneID', 'InteractionActions']).reset_index(drop=True)
if ctd_ixns.empty:
    raise ValueError('CTD reference is empty for these baseline PMIDs. Verify the CTD file and PMIDs.')
print(f'CTD reference interactions: {len(ctd_ixns):,}; PMIDs: {ctd_ixns.pmid.nunique()}')


## Gene normalization (from combined notebook)

In [ ]:
from __future__ import annotations

import csv
import gzip
import re
import warnings
from collections import defaultdict
from io import StringIO
from pathlib import Path
from urllib.parse import urlencode

import pandas as pd
import requests

SPECIES_TAXIDS = {"hsa": 9606, "rno": 10116, "mmu": 10090}
TAXID_SPECIES = {v: k for k, v in SPECIES_TAXIDS.items()}
NCBI_FILES = {9606: "Homo_sapiens.gene_info.gz", 10116: "Rattus_norvegicus.gene_info.gz", 10090: "Mus_musculus.gene_info.gz"}
NCBI_ROOT = "https://ftp.ncbi.nlm.nih.gov/gene/DATA/GENE_INFO/Mammalia/"
HGNC_URL = "https://storage.googleapis.com/public-download-files/hgnc/tsv/tsv/hgnc_complete_set.txt"
UNIPROT_ENDPOINT = "https://rest.uniprot.org/uniprotkb/stream"
CACHE_DIR = PROJECT_ROOT / "mirgenedb_normalization_reports" / "reference_data"

# Biological aliases are NOT hardcoded.
# Gene aliases come from NCBI Gene, HGNC, and UniProt.
# Human/rat/mouse taxonomy codes and data URLs remain configuration constants.

_SESSION = requests.Session()
_REFERENCE_CACHE = {}


def _clean(v):
    if v is None:
        return ""
    try:
        if bool(pd.isna(v)):
            return ""
    except (TypeError, ValueError):
        pass
    s = str(v).strip()
    return "" if s.lower() in {"", "nan", "none", "<na>", "null", "-"} else s


# Preserve Greek-letter meaning during gene-name normalization
GREEK = str.maketrans({
    "α": "alpha",
    "β": "beta",
    "γ": "gamma",
    "δ": "delta",
    "κ": "kappa",
})

def _key(v):
    s = _clean(v).casefold().translate(GREEK)
    return re.sub(r"[^a-z0-9]", "", s)


def _id(v):
    return re.sub(r"\.0$", "", _clean(v))


def _species_from_text(v):
    m = re.match(r"^(hsa|rno|mmu)[-_]", _clean(v), re.I)
    return m.group(1).lower() if m else ""


def _download(url, path, *, min_bytes=20):
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and path.stat().st_size >= min_bytes:
        return path
    temp = path.with_suffix(path.suffix + ".tmp")
    try:
        with _SESSION.get(url, stream=True, timeout=(20, 120), headers={"User-Agent": "CLOVER-normalizer/4.0"}) as r:
            r.raise_for_status()
            with temp.open("wb") as out:
                for chunk in r.iter_content(1024 * 1024):
                    if chunk:
                        out.write(chunk)
        if temp.stat().st_size < min_bytes:
            raise ValueError(f"Empty or truncated download: {url}")
        if path.suffix == ".gz":
            with gzip.open(temp, "rb") as f:
                f.read(32)
        temp.replace(path)
        return path
    finally:
        temp.unlink(missing_ok=True)


def _put(mapping, taxid, label, gid):
    k = _key(label)
    if k and gid:
        mapping[(taxid, k)].add(gid)


def _load_ncbi(cache_dir):
    by_id = {}
    official = defaultdict(set)
    synonyms = defaultdict(set)
    descriptions = defaultdict(set)
    for tid, fn in NCBI_FILES.items():
        path = _download(NCBI_ROOT + fn, cache_dir / fn)
        with gzip.open(path, "rt", encoding="utf-8", errors="replace") as f:
            header = next(f).lstrip("#").rstrip("\n").split("\t")
            columns = {name: i for i, name in enumerate(header)}
            required = {"tax_id", "GeneID", "Symbol", "Synonyms"}
            if not required <= columns.keys():
                raise ValueError(f"Invalid NCBI gene_info header: {path}")
            for line in f:
                parts = line.rstrip("\n").split("\t")
                if len(parts) < len(header) or parts[columns["tax_id"]] != str(tid):
                    continue
                gid, sym = _id(parts[columns["GeneID"]]), _clean(parts[columns["Symbol"]])
                if not gid or not sym:
                    continue
                desc = _clean(parts[columns["description"]]) if "description" in columns else ""
                by_id[(tid, gid)] = (sym, desc or sym)
                _put(official, tid, sym, gid)
                _put(descriptions, tid, desc, gid)
                for synonym in parts[columns["Synonyms"]].split("|"):
                    _put(synonyms, tid, synonym, gid)
    return by_id, official, synonyms, descriptions


def _load_hgnc(cache_dir, by_id):
    aliases = defaultdict(set)
    descriptions = defaultdict(set)
    path = _download(HGNC_URL, cache_dir / "hgnc_complete_set.txt", min_bytes=200)
    with path.open(encoding="utf-8", errors="replace", newline="") as f:
        for row in csv.DictReader(f, delimiter="\t"):
            if _clean(row.get("status")).lower() not in {"", "approved"}:
                continue
            gid = _id(row.get("entrez_id"))
            if (9606, gid) not in by_id:
                continue
            for col in ("symbol", "prev_symbol", "alias_symbol", "name", "prev_name", "alias_name"):
                vals = _clean(row.get(col)).split("|")
                dest = descriptions if col.endswith("name") else aliases
                for val in vals:
                    _put(dest, 9606, val, gid)
    return aliases, descriptions


def _uniprot_url(taxid):
    # Reviewed entries limit noisy aliases and isoform ambiguity.
    params = {
        "query": f"(organism_id:{taxid}) AND (reviewed:true)",
        "format": "tsv",
        "fields": "accession,gene_primary,gene_synonym,protein_name",
        "compressed": "true",
    }
    return UNIPROT_ENDPOINT + "?" + urlencode(params)


def _load_uniprot(cache_dir, by_id, official, synonyms, descriptions):
    """Map UniProt alternative protein names via uniquely grounded gene names.

    Only protein names with a unique species-specific NCBI gene are eligible.
    Conflicting UniProt names are stored as multi-ID and rejected downstream.
    """
    mapping = defaultdict(set)
    for taxid in SPECIES_TAXIDS.values():
        path = _download(_uniprot_url(taxid), cache_dir / f"uniprot_reviewed_{taxid}.tsv.gz", min_bytes=200)
        with gzip.open(path, "rt", encoding="utf-8", errors="replace", newline="") as f:
            rows = csv.DictReader(f, delimiter="\t")
            for row in rows:
                primary = _clean(row.get("Gene Names (primary)"))
                alternatives = _clean(row.get("Gene Names (synonym)"))
                candidates = set()
                for name in [primary] + alternatives.split():
                    if not name:
                        continue
                    k = (taxid, _key(name))
                    for pool in (official, synonyms):
                        ids = pool.get(k, set())
                        if len(ids) == 1:
                            candidates.update(ids)
                            break
                if len(candidates) != 1:
                    continue
                gid = next(iter(candidates))
                protein = _clean(row.get("Protein names"))
                # UniProt names commonly format: Recommended Name: Full=...;
                # Alternative Name: Full=...; Short=...;
                names = re.findall(r"(?:Full|Short)=([^;\[\]]+)", protein)
                if not names and protein:
                    names = [protein.split(" (")[0]]
                for name in names:
                    name = re.sub(r"\s*\{[^}]*\}\s*$", "", name).strip()
                    if len(name) > 2:
                        _put(mapping, taxid, name, gid)
    return mapping


def _references(cache_dir, use_hgnc=True, use_uniprot=True):
    signature = (str(cache_dir.resolve()), use_hgnc, use_uniprot)
    if signature in _REFERENCE_CACHE:
        return _REFERENCE_CACHE[signature]
    by_id, official, synonyms, descriptions = _load_ncbi(cache_dir)
    # Keep per-reference lookup tables separate *before* merging aliases.
    # The existing normalization priorities and candidate pools are unchanged.
    ncbi_aliases = {k: set(v) for k, v in synonyms.items()}
    ncbi_descriptions = {k: set(v) for k, v in descriptions.items()}
    hgnc_aliases, hgnc_descriptions = defaultdict(set), defaultdict(set)
    extras = defaultdict(set)
    if use_hgnc:
        try:
            hg_alias, hg_desc = _load_hgnc(cache_dir, by_id)
            hgnc_aliases, hgnc_descriptions = hg_alias, hg_desc
            for k, v in hg_alias.items():
                synonyms[k].update(v)
            for k, v in hg_desc.items():
                descriptions[k].update(v)
        except Exception as exc:
            warnings.warn(f"HGNC unavailable; using NCBI only for these aliases: {exc}")
    if use_uniprot:
        try:
            extras = _load_uniprot(cache_dir, by_id, official, synonyms, descriptions)
        except Exception as exc:
            warnings.warn(f"UniProt unavailable; using NCBI/HGNC aliases: {exc}")
    provenance = {
        "ncbi_aliases": ncbi_aliases,
        "ncbi_descriptions": ncbi_descriptions,
        "hgnc_aliases": hgnc_aliases,
        "hgnc_descriptions": hgnc_descriptions,
    }
    _REFERENCE_CACHE[signature] = (by_id, official, synonyms, descriptions, extras, provenance)
    return _REFERENCE_CACHE[signature]


def _pmid_species(pmid_species, pmid):
    if pmid_species is None:
        return ""
    p = _id(pmid)
    val = pmid_species.get(p, pmid_species.get(int(p)) if p.isdigit() else None)
    if val is None:
        return ""
    vals = [val] if isinstance(val, str) else list(val)
    matches = {str(v).lower() for v in vals} & set(SPECIES_TAXIDS)
    return next(iter(matches)) if len(matches) == 1 else ("ambiguous" if matches else "")


# ---------------------------------------------------------------------------
# MirGeneDB 3.x: cached browse table + GFF loci, mapped to NCBI Gene
# ---------------------------------------------------------------------------
_MIR_CACHE = {}

def _is_mirna(v):
    s = _clean(v)
    return bool(re.match(r"^(?:(?:hsa|rno|mmu)[-_])?(?:mir(?:na|cro-?rna)?|let)[-_]?\d", s, re.I)) or bool(re.match(r"^MIRLET\d", s, re.I))

def _mir_key(v):
    return re.sub(r"[^a-z0-9]", "", _clean(v).casefold())

def _mir_names(raw, species):
    """Conservative precursor-name candidates; mature arm is checked separately."""
    s = re.sub(r"^(hsa|rno|mmu)[-_]", "", _clean(raw), flags=re.I).lower()
    arm_match = re.search(r"[-_](3p|5p)(?:\*)?$", s)
    arm = arm_match.group(1) if arm_match else ""
    if arm_match:
        s = s[:arm_match.start()]
    s = re.sub(r"^microrna[-_]?", "mir-", s)
    s = re.sub(r"^mirna[-_]?", "mir-", s)
    if s.startswith("mirlet"):
        s = "let-" + s[6:]
    if s.startswith("mir") and re.match(r"^mir\d", s):
        s = "mir-" + s[3:]
    return {_mir_key(species + '-' + s)}, arm

def _mir_locus_key(v):
    s = re.sub(r"_(?:pre|[35]p)\*?$", "", _clean(v), flags=re.I)
    s = re.sub(r"-v\d+$", "", s, flags=re.I)
    return _mir_key(s)

def _load_mirgenedb(cache_dir, species, by_id, official, synonyms):
    signature = (str(cache_dir.resolve()), species)
    if signature in _MIR_CACHE:
        return _MIR_CACHE[signature]
    browse_url = f"https://mirgenedb.org/browse/{species}"
    gff_url = f"https://mirgenedb.org/gff/{species}?all=1&sort=pos"
    browse_path = _download(browse_url, cache_dir / f"mirgenedb_{species}_browse.html", min_bytes=100)
    gff_path = _download(gff_url, cache_dir / f"mirgenedb_{species}.gff3", min_bytes=100)
    tables = pd.read_html(StringIO(browse_path.read_text(encoding="utf-8", errors="replace")))
    browse = None
    for table in tables:
        cols = [" ".join(map(str, c)) if isinstance(c, tuple) else str(c) for c in table.columns]
        mir_cols = [i for i, c in enumerate(cols) if "mirgenedb" in c.lower() and "id" in c.lower()]
        base_cols = [i for i, c in enumerate(cols) if "mirbase" in c.lower() and "id" in c.lower()]
        if mir_cols and base_cols:
            browse = table.iloc[:, [mir_cols[0], base_cols[0]]].dropna()
            break
    if browse is None:
        raise ValueError(f"MirGeneDB browse columns not found for {species}")

    loci = defaultdict(lambda: {"pre": set(), "arms": set()})
    for line in gff_path.read_text(encoding="utf-8", errors="replace").splitlines():
        if not line or line.startswith("#"):
            continue
        parts = line.split("\t")
        if len(parts) < 9 or parts[2] not in {"pre_miRNA", "miRNA"}:
            continue
        attrs = dict(v.split("=", 1) for v in parts[8].split(";") if "=" in v)
        fid = attrs.get("ID", "")
        if not fid:
            continue
        locus = loci[_mir_locus_key(fid)]
        if parts[2] == "pre_miRNA":
            locus["pre"].add(fid)
        else:
            m = re.search(r"_([35]p)\*?$", fid, re.I)
            if m:
                locus["arms"].add(m.group(1).lower())

    tid = SPECIES_TAXIDS[species]
    index = defaultdict(dict)
    # Build a canonical NCBI lookup strictly from symbols/synonyms, no fuzzy mapping.
    for mirgene, precursor in browse.itertuples(index=False, name=None):
        mirgene, precursor = _clean(mirgene), _clean(precursor)
        locus = loci.get(_mir_locus_key(mirgene))
        if not locus or not precursor:
            continue
        base_key, _ = _mir_names(precursor, species)
        keys = set(base_key)
        # mirbase precursor usually already contains prefix and may have an arm.
        keys.add(_mir_key(precursor))
        local = re.sub(rf"^{species}[-_]", "", precursor, flags=re.I)
        ncbi_keys = {_key(local)}
        if local.lower().startswith('let-'):
            ncbi_keys.add(_key('mir' + local))
        if local.lower().startswith('mir-'):
            ncbi_keys.add(_key('mir' + local[4:]))
        ids = set()
        for k in ncbi_keys:
            ids.update(official.get((tid, k), set()))
            ids.update(synonyms.get((tid, k), set()))
        ids = {v for v in ids if (tid, v) in by_id}
        for key in keys:
            index[key][mirgene] = (set(ids), set(locus['arms']))
    _MIR_CACHE[signature] = index
    return index

def _mir_resolve(raw, species, mir_index):
    keys, arm = _mir_names(raw, species)
    matched = {}
    for key in keys:
        matched.update(mir_index.get(key, {}))
    if arm:
        matched = {locus: val for locus, val in matched.items() if arm in val[1]}
    if not matched:
        return set(), 'mirgenedb_unresolved'
    if len(matched) != 1:
        return set(), 'mirgenedb_ambiguous_locus'
    ids = next(iter(matched.values()))[0]
    if len(ids) != 1:
        return set(), 'mirgenedb_no_unique_ncbi_gene'
    return ids, 'mirgenedb_unique'


def normalize_gene_symbols(df, cache_dir=CACHE_DIR, pmid_species=None,
                           default_species="hsa", strict_conflicts=True,
                           use_hgnc=True, use_uniprot=True, use_mirgenedb=True):
    """Return df with unique, species-aware official gene symbols.

    Priorities: prior verified miRNA > MirGeneDB precursor locus + NCBI Gene
    > official NCBI symbol > NCBI/HGNC alias
    > NCBI/HGNC gene description > UniProt protein name.
    Each mapping must be unique and species-consistent.
    Gene IDs are used for validation, not to choose the target when ambiguous.
    Does not write audit files.
    """
    df = df.copy()
    if "geneSymbol" not in df.columns:
        return df
    for col in ("geneID", "geneName"):
        if col not in df.columns:
            df[col] = ""
    for col in ("geneSymbol", "geneID", "geneName"):
        if col + "_original" not in df.columns:
            df[col + "_original"] = df[col]
    if "normalization_status" not in df.columns:
        df["normalization_status"] = "unchanged"

    # PMID-aware inference from explicit species-prefixed miRNAs in this DataFrame.
    # Caller-supplied pmid_species takes precedence. Mixed species remain ambiguous.
    if pmid_species is None and "pmid" in df.columns:
        inferred_species = defaultdict(set)
        for pmid_value, symbol_value in df[["pmid", "geneSymbol"]].itertuples(index=False, name=None):
            if _is_mirna(symbol_value):
                explicit = _species_from_text(symbol_value)
                if explicit:
                    inferred_species[_id(pmid_value)].add(explicit)
        pmid_species = inferred_species

    by_id, official, aliases, descriptions, proteins, provenance = _references(
        Path(cache_dir), use_hgnc, use_uniprot)
    out = {c: df[c].tolist() for c in ("geneSymbol", "geneID", "geneName", "normalization_status")}
    statuses, verified, species_list = [], [], []
    resolution_db_trace = [""] * len(df)
    mir_indexes = {}
    # Lazy initialization: only download MirGeneDB for species encountered.
    # A failed optional download retains other references but never claims a MirGeneDB match.
    mir_failures = set()

    for i, (_, row) in enumerate(df.iterrows()):
        symbol = _clean(row.get("geneSymbol"))
        name = _clean(row.get("geneName"))
        gid = _id(row.get("geneID"))
        status, is_verified, species = "unchanged", False, ""
        if not symbol and not name:
            statuses.append(status); verified.append(False); species_list.append("")
            continue

        mir_status = _clean(row.get("mirna_mapping_status")).lower()
        previous_status = _clean(row.get("normalization_status")).lower()
        valid_mir = (previous_status == "high_confidence_normalized" and gid and
                     (mir_status in {"high_confidence", "applied", "mapped", "verified",
                                     "high_confidence_normalized", "unique", "mirgenedb_unique"}
                      or (_is_mirna(row.get("geneSymbol_original")) and mir_status not in {
                          "", "unresolved", "ambiguous", "secondary_unique_candidate",
                          "not_mirna_or_not_identified", "not_found"})))
        if valid_mir:
            statuses.append("existing_mirna_normalization")
            verified.append(True); species_list.append(_species_from_text(symbol))
            continue

        exp_sym = _species_from_text(symbol)
        exp_name = _species_from_text(name)
        if exp_sym and exp_name and exp_sym != exp_name:
            statuses.append("species_conflict_review"); verified.append(False); species_list.append("")
            continue
        species = exp_sym or exp_name
        pmid_sp = _pmid_species(pmid_species, row.get("pmid"))
        if pmid_sp == "ambiguous" and not species:
            statuses.append("ambiguous_species"); verified.append(False); species_list.append("")
            continue
        if species and pmid_sp not in {"", "ambiguous", species}:
            statuses.append("species_conflict_review"); verified.append(False); species_list.append(species)
            continue
        if not species:
            # Existing Gene ID can independently establish the organism; never force human otherwise.
            found = {tid for tid in SPECIES_TAXIDS.values() if (tid, gid) in by_id} if gid else set()
            if len(found) == 1:
                species = TAXID_SPECIES[next(iter(found))]
            elif len(found) > 1 and not pmid_sp:
                statuses.append("ambiguous_species"); verified.append(False); species_list.append("")
                continue
            else:
                species = pmid_sp or default_species
        if species not in SPECIES_TAXIDS:
            statuses.append("unsupported_species"); verified.append(False); species_list.append(species)
            continue
        tid = SPECIES_TAXIDS[species]
        local_symbol = re.sub(r"^(hsa|rno|mmu)[-_]", "", symbol, flags=re.I).strip()
        ksym = _key(local_symbol)
        is_mir = _is_mirna(symbol) or _is_mirna(name)
        if is_mir and use_mirgenedb:
            if species not in mir_indexes and species not in mir_failures:
                try:
                    mir_indexes[species] = _load_mirgenedb(
                        Path(cache_dir), species, by_id, official, aliases)
                except Exception as exc:
                    warnings.warn(f"MirGeneDB unavailable for {species}: {exc}")
                    mir_failures.add(species)
            if species in mir_indexes:
                mir_candidates, mir_state = _mir_resolve(symbol or name, species, mir_indexes[species])
                # A known ambiguous locus is not disambiguated by unrelated protein aliases.
                if mir_state == "mirgenedb_ambiguous_locus":
                    statuses.append(mir_state); verified.append(False); species_list.append(species)
                    continue
                if len(mir_candidates) == 1:
                    target = next(iter(mir_candidates))
                    if gid and gid != target and strict_conflicts:
                        statuses.append("gene_id_conflict_review"); verified.append(False); species_list.append(species)
                        continue
                    proposed, desc = by_id[(tid, target)]
                    if proposed != symbol:
                        out["geneSymbol"][i] = proposed
                    if not gid:
                        out["geneID"][i] = target
                    if not name or _key(name) == _key(symbol):
                        out["geneName"][i] = desc
                    if proposed != symbol or not gid:
                        out["normalization_status"][i] = "high_confidence_normalized"
                    statuses.append("mapped_mirgenedb" if proposed != symbol else "already_official_mirna")
                    verified.append(True); species_list.append(species)
                    continue
                # Fall through to exact NCBI aliases when no unique MirGeneDB locus.
        # Do not merge candidates from independent name fields. A resolved symbol wins;
        # the name is a fallback and only a separately resolved contradiction triggers review.
        candidates, source = set(), ""
        resolution_key = ksym
        for src, mapping in (("official", official), ("synonym", aliases),
                             ("gene_description", descriptions), ("uniprot_protein", proteins)):
            found = mapping.get((tid, ksym), set())
            if found:
                candidates, source = set(found), src
                break
        if not candidates and name:
            kname = _key(name)
            for src, mapping in (("name_official", official), ("name_synonym", aliases),
                                 ("name_description", descriptions), ("name_uniprot", proteins)):
                found = mapping.get((tid, kname), set())
                if found:
                    candidates, source = set(found), src
                    resolution_key = kname
                    break
        if len(candidates) != 1:
            status = "ambiguous_reference_mapping" if candidates else "unresolved"
            statuses.append(status); verified.append(False); species_list.append(species)
            continue
        target = next(iter(candidates))
        if gid and gid != target and strict_conflicts:
            statuses.append("gene_id_conflict_review"); verified.append(False); species_list.append(species)
            continue
        if name and _key(name) != ksym:
            # Name contradictions are checked, but absence from reference is not a veto.
            kname = _key(name)
            name_ids = set()
            for mapping in (official, aliases, descriptions):
                name_ids = mapping.get((tid, kname), set())
                if name_ids:
                    break
            if name_ids and target not in name_ids and source != "official":
                statuses.append("symbol_name_conflict_review"); verified.append(False); species_list.append(species)
                continue
        # Resolve precisely which reference(s) support the *selected* alias or name.
        if source in {"synonym", "name_synonym", "gene_description", "name_description"}:
            key_prefix = "aliases" if "synonym" in source else "descriptions"
            ncbi_pool = provenance[f"ncbi_{key_prefix}"]
            hgnc_pool = provenance[f"hgnc_{key_prefix}"]
            in_ncbi = target in ncbi_pool.get((tid, resolution_key), set())
            in_hgnc = target in hgnc_pool.get((tid, resolution_key), set())
            if in_ncbi and in_hgnc:
                resolution_db_trace[i] = "NCBI Gene + HGNC (both support alias)"
            elif in_hgnc:
                resolution_db_trace[i] = "HGNC"
            elif in_ncbi:
                resolution_db_trace[i] = "NCBI Gene"
        proposed, desc = by_id[(tid, target)]
        is_verified = True
        symbol_changed = symbol != proposed
        if symbol_changed:
            out["geneSymbol"][i] = proposed
        if not gid:
            out["geneID"][i] = target
        if not name or _key(name) == _key(symbol):
            out["geneName"][i] = desc
        if symbol_changed or not gid:
            out["normalization_status"][i] = "high_confidence_normalized"
        status = f"mapped_{source}" if symbol_changed else "already_official_symbol"
        statuses.append(status); verified.append(is_verified); species_list.append(species)

    for c, values in out.items():
        df[c] = values
    df["gene_symbol_resolution"] = statuses
    df["gene_identity_verified"] = verified
    df["gene_symbol_species"] = species_list
    # Provenance of the accepted *second-stage* match; unchanged validations
    # are included and should not be confused with fields that were changed.
    def _db_from_status(st):
        if st in {"mapped_mirgenedb", "already_official_mirna"}:
            return "MirGeneDB + NCBI Gene"
        if st == "existing_mirna_normalization":
            return "Previously verified miRNA (source not recorded here)"
        if st in {"mapped_official", "mapped_name_official", "already_official_symbol"}:
            return "NCBI Gene"
        if "uniprot" in st:
            return "UniProt (NCBI Gene anchored)"
        if st.startswith(("mapped_",)):
            return "Unknown merged alias provenance"
        return "No accepted mapping (or unresolved)"
    df["gene_resolution_database"] = [
        trace if trace else _db_from_status(status)
        for trace, status in zip(resolution_db_trace, statuses)
    ]
    if "gene_key" in df.columns:
        df = df.drop(columns=["gene_key"])
    print("Gene symbols: changed=", sum(a != b for a, b in zip(df["geneSymbol"], df["geneSymbol_original"])),
          "verified=", sum(verified), "ambiguous/conflicting=",
          sum(s.startswith(("ambiguous", "gene_id_conflict", "symbol_name_conflict")) for s in statuses))
    return df


In [ ]:
# Copied from Bedrock_preAnalysis section 8; helper logic unchanged.
# Intended order within the combined analysis:
#   update_dataframe -> normalize_gene_symbols (external DBs)
#   -> update_gene_symbol_ (custom CTD fallback) -> dedup -> optional filter
import re
from typing import Optional
import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Small pure helpers
# ---------------------------------------------------------------------------

def _symbol_numeric_suffix(symbol) -> Optional[str]:
    """
    Extract the portion of a GeneSymbol starting from its first digit.
        FKBP1A -> "1A"
        IL6    -> "6"
        TP53   -> "53"
    Returns None if no digit is found or input is NaN.
    """
    if pd.isna(symbol):
        return None
    m = re.search(r"\d.*", str(symbol))
    return m.group(0) if m else None


def _tokenize(name) -> list:
    """Split a (already-cleaned) name string into lowercase tokens."""
    if pd.isna(name) or not str(name).strip():
        return []
    return str(name).strip().lower().split()


def _remove_parentheses(name) -> str:
    """
    Strip parenthesised substrings (including the parentheses themselves).
        "glyceraldehyde-3-phosphate dehydrogenase (GAPDH)"
        -> "glyceraldehyde-3-phosphate dehydrogenase"
    """
    if pd.isna(name):
        return ""
    return re.sub(r"\s*\(.*?\)", "", str(name)).strip()


def _suffix_in_tokens(suffix: Optional[str], tokens: list) -> bool:
    """
    Return True when *suffix* (e.g. "53") is represented in *tokens*.

    Single-token names  : suffix must appear *anywhere* in the token.
    Multi-token names   : at least one token must *end with* suffix.
    """
    if not suffix:
        return False
    suffix = suffix.lower()
    if len(tokens) == 1:
        return suffix in tokens[0]
    return any(tok.endswith(suffix) for tok in tokens)


def _normalize(s) -> str:
    """Lowercase, strip, coerce NaN -> empty string."""
    if pd.isna(s):
        return ""
    return str(s).strip().lower()


def _alt_count(alt_ids_str) -> int:
    """Count non-empty pipe-separated alternative gene IDs."""
    return len([a for a in str(alt_ids_str).split("|") if a.strip()])


def _symbol_is_missing(symbol) -> bool:
    """Return True if a geneSymbol value should be treated as absent."""
    if pd.isna(symbol):
        return True
    return str(symbol).strip() in ("", "-")


def _strip_dot_suffix(symbol) -> Optional[str]:
    """
    If a symbol contains a dot, return the portion before it.
        BRCA1.2  -> "BRCA1"
        TP53.1   -> "TP53"
        BRCA1    -> None  (no dot present)
    """
    if pd.isna(symbol):
        return None
    s = str(symbol).strip()
    if "." in s:
        return s.split(".")[0]
    return None


# ---------------------------------------------------------------------------
# CTD pre-processing  (call once, reuse across batches)
# ---------------------------------------------------------------------------

def prepare_ctd(ctd_genes: pd.DataFrame) -> dict:
    """
    Build fast lookup structures from the CTD gene table.

    Returns a dict with:
        "df"           : cleaned CTD dataframe with helper columns
        "id_index"     : {gene_id_str          -> list[row_index]}
        "symbol_index" : {normalised_symbol    -> list[row_index]}
        "name_index"   : {normalised_gene_name -> list[row_index]}
        "synonym_index": {normalised_synonym   -> list[row_index]}
    """
    ctd = ctd_genes.copy()
    ctd["GeneID"]       = ctd["GeneID"].astype(str)
    ctd["_symbol_norm"] = ctd["# GeneSymbol"].apply(_normalize)
    ctd["_name_norm"]   = ctd["GeneName"].apply(_normalize)
    ctd["_alt_count"]   = ctd["AltGeneIDs"].fillna("").apply(_alt_count)
    ctd["_syns"]        = ctd["Synonyms"].fillna("").str.lower().str.split("|")

    id_index: dict = {}
    for i, gid in ctd["GeneID"].items():
        if gid:
            id_index.setdefault(gid, []).append(i)

    symbol_index: dict = {}
    for i, sym in ctd["_symbol_norm"].items():
        if sym:
            symbol_index.setdefault(sym, []).append(i)

    name_index: dict = {}
    for i, name in ctd["_name_norm"].items():
        if name:
            name_index.setdefault(name, []).append(i)

    synonym_index: dict = {}
    for i, syns in ctd["_syns"].items():
        for s in syns:
            s = s.strip()
            if s:
                synonym_index.setdefault(s, []).append(i)

    return {
        "df": ctd,
        "id_index": id_index,
        "symbol_index": symbol_index,
        "name_index": name_index,
        "synonym_index": synonym_index,
    }


# ---------------------------------------------------------------------------
# Per-row resolution helpers
# ---------------------------------------------------------------------------

def _resolve_by_id(gene_id: str, ctd_lookup: dict) -> Optional[pd.Series]:
    """Priority 1: exact GeneID match."""
    rows = ctd_lookup["df"].loc[ctd_lookup["id_index"].get(gene_id, [])]
    return rows.iloc[0] if not rows.empty else None


def _resolve_by_symbol(symbol_norm: str, ctd_lookup: dict) -> Optional[pd.Series]:
    """Priority 2: exact normalised GeneSymbol match."""
    rows = ctd_lookup["df"].loc[ctd_lookup["symbol_index"].get(symbol_norm, [])]
    return rows.iloc[0] if not rows.empty else None


def _resolve_by_name(name_norm: str, name_tokens: list, ctd_lookup: dict) -> Optional[pd.Series]:
    """
    Priority 3: match via GeneName or Synonyms with alt_count tiebreaking.

    Cases
    -----
    1. Only GeneName hits  : prefer suffix-matching candidates; tiebreak by alt_count
    2. Both hits           : compare total alt_count of each group; pick winner's best
    3. Only Synonym hits   : pick highest alt_count
    """
    if not name_norm:
        return None

    ctd_df        = ctd_lookup["df"]
    name_index    = ctd_lookup["name_index"]
    synonym_index = ctd_lookup["synonym_index"]

    gn_rows  = ctd_df.loc[name_index.get(name_norm, [])]
    syn_rows = ctd_df.loc[synonym_index.get(name_norm, [])]

    # Deduplicate: remove synonym hits already present in gene-name hits
    if not gn_rows.empty and not syn_rows.empty:
        syn_rows = syn_rows[~syn_rows.index.isin(gn_rows.index)]

    # ---- Case 1: only GeneName hits ----
    if not gn_rows.empty and syn_rows.empty:
        suffix_matches = [
            r for _, r in gn_rows.iterrows()
            if _suffix_in_tokens(_symbol_numeric_suffix(r["# GeneSymbol"]), name_tokens)
        ]
        best_df = pd.DataFrame(suffix_matches) if suffix_matches else gn_rows
        return best_df.sort_values("_alt_count", ascending=False).iloc[0]

    # ---- Case 2: both hits ----
    if not gn_rows.empty and not syn_rows.empty:
        if gn_rows["_alt_count"].sum() >= syn_rows["_alt_count"].sum():
            return gn_rows.sort_values("_alt_count", ascending=False).iloc[0]
        else:
            return syn_rows.sort_values("_alt_count", ascending=False).iloc[0]

    # ---- Case 3: only Synonym hits ----
    if not syn_rows.empty:
        return syn_rows.sort_values("_alt_count", ascending=False).iloc[0]

    return None


# ---------------------------------------------------------------------------
# Dot-symbol resolution helper
# ---------------------------------------------------------------------------

def _apply_dot_check(best: pd.Series, ctd_lookup: dict) -> pd.Series:
    """
    If the resolved symbol contains a dot (e.g. BRCA1.2), strip the suffix
    and re-look up the base symbol in CTD to get the correct GeneID/GeneName.
    Returns the corrected row, or the original row with the dot stripped if
    no CTD entry is found for the base.
    """
    base = _strip_dot_suffix(best["# GeneSymbol"])
    if base is None:
        return best  # no dot, nothing to do

    hit = _resolve_by_symbol(_normalize(base), ctd_lookup)
    if hit is not None:
        return hit

    # No CTD entry for base symbol — strip the dot suffix in-place
    best = best.copy()
    best["# GeneSymbol"] = base
    return best


# ---------------------------------------------------------------------------
# Public entry point
# ---------------------------------------------------------------------------
def update_gene_symbol_(
    df: pd.DataFrame,
    ctd_genes: pd.DataFrame,
    ctd_lookup: Optional[dict] = None,
) -> pd.DataFrame:

    df = df.copy()

    df["oldGeneID"]     = df.get("geneID")
    df["oldGeneName"]   = df.get("geneName")
    df["oldGeneSymbol"] = df.get("geneSymbol")

    # ✅ Treat "-", " -", "- " etc. as missing across key gene columns
    for col in ["geneID", "geneSymbol", "geneName"]:
        if col in df.columns:
            df[col] = df[col].replace(r"^\s*-\s*$", "", regex=True).fillna("")

    if ctd_lookup is None:
        ctd_lookup = prepare_ctd(ctd_genes)

    # Vectorised pre-normalisation
    df["_name_norm"] = (
        df.get("geneName", pd.Series(dtype=str))
        .apply(_remove_parentheses)
        .apply(_normalize)
    )
    df["_symbol_norm"] = (
        df.get("geneSymbol", pd.Series(dtype=str))
        .apply(_normalize)
    )

    flags       = np.full(len(df), 2, dtype=int)
    new_symbols = df["geneSymbol"].copy() if "geneSymbol" in df.columns else pd.Series(index=df.index, dtype=str)
    new_ids     = df["geneID"].astype(str).copy()
    new_names   = df["geneName"].copy()   if "geneName"   in df.columns else pd.Series(index=df.index, dtype=str)

    for pos, (idx, row) in enumerate(df.iterrows()):
        raw_symbol  = row.get("geneSymbol")
        gene_id     = str(row.get("geneID", "")).strip()
        symbol_norm = row["_symbol_norm"]
        name_norm   = row["_name_norm"]
        name_tokens = _tokenize(name_norm)

        # Already valid symbol
        if not _symbol_is_missing(raw_symbol):
            if not gene_id:  # symbol present but ID missing — try to resolve ID from symbol
                best = _resolve_by_symbol(symbol_norm, ctd_lookup)
                if best is not None:
                    best = _apply_dot_check(best, ctd_lookup)
                    new_symbols.iloc[pos] = best["# GeneSymbol"]
                    new_ids.iloc[pos]     = best["GeneID"]
                    new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0
            continue
    
        # Priority 1: GeneID
        if gene_id:
            best = _resolve_by_id(gene_id, ctd_lookup)
            if best is not None:
                best = _apply_dot_check(best, ctd_lookup)
                new_symbols.iloc[pos] = best["# GeneSymbol"]
                new_ids.iloc[pos]     = best["GeneID"]
                new_names.iloc[pos]   = best["GeneName"]
                flags[pos] = 0
                continue

        # Priority 2: GeneSymbol
        if symbol_norm:
            best = _resolve_by_symbol(symbol_norm, ctd_lookup)
            if best is not None:
                best = _apply_dot_check(best, ctd_lookup)
                new_symbols.iloc[pos] = best["# GeneSymbol"]
                new_ids.iloc[pos]     = best["GeneID"]
                new_names.iloc[pos]   = best["GeneName"]
                flags[pos] = 0
                continue

        # Priority 3a: treat geneName as a symbol
        best = _resolve_by_symbol(name_norm, ctd_lookup)
        if best is not None:
            best = _apply_dot_check(best, ctd_lookup)
            new_symbols.iloc[pos] = best["# GeneSymbol"]
            new_ids.iloc[pos]     = best["GeneID"]
            new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0
            continue

        # Priority 3b: GeneName / Synonym
        best = _resolve_by_name(name_norm, name_tokens, ctd_lookup)
        if best is not None:
            best = _apply_dot_check(best, ctd_lookup)
            new_symbols.iloc[pos] = best["# GeneSymbol"]
            new_ids.iloc[pos]     = best["GeneID"]
            new_names.iloc[pos]   = best["GeneName"]
            flags[pos] = 0

    df["geneSymbol"]     = new_symbols.values
    df["geneID"]         = new_ids.values
    df["geneName"]       = new_names.values
    df["geneSymbolFlag"] = flags

    df.drop(columns=["_name_norm", "_symbol_norm"], inplace=True)
    return df


## Standardization, baseline-compatible filtering, and CTD matching


In [ ]:
def update_dataframe(sample):
    """
    Normalize DataFrame columns without removing rows.
    """

    sample = sample.copy()

    # Normalize text columns
    string_cols = [
        "source",
        "text",
        "relationship",
        "chemicalName",
        "chemicalID",
        "geneSymbol",
        "geneID",
        "section_type",
        "chemical_map",
        "gene_map",
        "reasoning",
    ]

    for col in string_cols:
        if col not in sample.columns:
            sample[col] = ""

        sample[col] = (
            sample[col]
            .fillna("")
            .astype(str)
            .str.strip()
        )

    # Normalize PMID
    sample["pmid"] = (
        sample["pmid"]
        .astype(str)
        .str.replace(r"\.0$", "", regex=True)
        .str.strip()
    )

    # Normalize section type
    sample["section_type"] = (
        sample["section_type"]
        .str.upper()
        .str.strip()
    )

    # Normalize chemical IDs
    sample["chemicalID"] = (
        sample["chemicalID"]
        .str.upper()
        .replace({
            "D000072317": "D013749"
        })
    )

    # CASE-INSENSITIVE GENE SYMBOL MATCHING
    # Preserve geneSymbol_original if it exists.
    sample["geneSymbol"] = (
        sample["geneSymbol"]
        .astype("string")
        .str.strip()
        .str.upper()
    )

    # Preserve original relationship
    sample["relationship_original"] = sample["relationship"]

    # Normalize relationship formatting
    sample["relationship"] = (
        sample["relationship"]
        .str.lower()
        .str.replace("^", " ", regex=False)
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )

    # Sort if original index exists
    if "Unnamed: 0" in sample.columns:
        sample = sample.sort_values("Unnamed: 0")

    return sample.reset_index(drop=True)


In [ ]:
def normalize_relationship(s):
    if pd.isna(s):
        return ''
    s = str(s).lower().strip().replace('^', ' ')
    s = s.replace('increases', 'increase').replace('decreases', 'decrease').replace('affects', 'affect')
    return ' '.join(s.split())


def dedup_baseline(df):
    # Same normalized CTD match keys as the combined notebook, with no section priority.
    out = df.copy()
    out['relationship_norm'] = out['relationship'].apply(normalize_relationship)
    out['gene_key'] = (out['geneSymbol'].fillna('').astype(str).str.strip().str.upper()
                       .str.replace(r'\..+$', '', regex=True))
    fallback = pd.to_numeric(out['geneID'], errors='coerce').astype('Int64').astype('string')
    out['gene_key'] = out['gene_key'].mask(out['gene_key'].isin(['', 'NAN', 'NA', '<NA>']), fallback)
    return out.drop_duplicates(subset=['pmid','chemicalID','gene_key','relationship_norm'], keep='first').copy()


def filter_relationships(group):
    # Same action-specific removal of generic "affect" when a directional action exists.
    part = group.copy()
    parsed = part['relationship'].apply(normalize_relationship).str.extract(r'^(increase|decrease|affect)\s+(.+)$')
    specific = set(parsed.loc[parsed[0].isin(['increase','decrease']), 1].dropna())
    return part.loc[~(parsed[0].eq('affect') & parsed[1].isin(specific))].copy()


def filter_baseline(df, chemicals, genes):
    """The combined notebook's filters that are applicable to a sectionless PDF baseline."""
    out = df.copy()
    out['chemicalID'] = out['chemicalID'].fillna('').astype(str).str.strip().str.upper()
    out['geneSymbol'] = out['geneSymbol'].fillna('').astype(str).str.strip().str.upper()
    out = out[out['chemicalID'].str.fullmatch(r'[CD]\d+', na=False)].copy()
    if not out.empty:
        out = pd.concat([filter_relationships(g) for _, g in
                         out.groupby(['pmid','chemicalID','geneSymbol'], dropna=False, sort=False)],
                        ignore_index=True)

    chem_vocab = set(chemicals['# ChemicalName'].dropna().astype(str).str.strip().str.lower())
    gene_vocab = set(pd.concat([genes['# GeneSymbol'], genes['GeneName']])
                     .dropna().astype(str).str.strip().str.upper())
    out = out[
        out['chemicalName'].fillna('').astype(str).str.strip().str.lower().isin(chem_vocab)
        & out['geneSymbol'].isin(gene_vocab)
    ].copy()

    # Preserve the entity/relationship flag filtering from the combined notebook.
    keep = pd.Series(True, index=out.index)
    for c in ['chemicalFlag','relationshipWarningFlag','prefixMissingFlag']:
        keep &= pd.to_numeric(out[c], errors='coerce').eq(0)
    valid_gene_flag = pd.to_numeric(out['geneSymbolFlag'], errors='coerce').eq(0)
    high_confidence = out['normalization_status'].fillna('').astype(str).eq('high_confidence_normalized')
    valid_gene_id = out['geneID'].fillna('').astype(str).str.strip().str.replace(r'\.0$', '', regex=True).str.fullmatch(r'\d+')
    keep &= valid_gene_flag | (high_confidence & valid_gene_id.fillna(False))
    out = out.loc[keep].copy()

    # Same baseline-applicable source heuristics, without section/text/caption checks.
    banned_source = (
        r'\b(?:might|maybe|perhaps|potentially|possibly|conceivably|plausibly|hypothesized|speculated)\b'
        r'|\bno significant (?:change|effect|difference)\b|\bnot statistically significant\b'
        r'|\b(?:did|does) not (?:affect|alter|change|induce)\b|\bno (?:effect|change|difference)\b'
        r'|\bunchanged\b|\bno observable effect\b|\bfailed to demonstrate\b'
        r'|\bnot associated with\b|\bno correlation\b|\bno relationship\b'
        r'|\bno impact\b|\bno influence\b|\bwas ineffective\b'
        r'|\bit is known\b|\bwidely believed\b|\bprevious research\b'
        r'|\baccording to studies\b|\breported previously\b|\bnumerous studies\b'
        r'|\bit has been suggested\b|\bgenerally accepted\b|\bwidely recognized\b'
    )
    out = out.loc[~out['source'].fillna('').astype(str).str.contains(banned_source, case=False, regex=True)].copy()
    out = out.loc[out['relationship'].apply(normalize_relationship).str.match(r'^(increase|decrease|affect)\s+\S+')].copy()
    return out.reset_index(drop=True)


In [ ]:
def matching_keys(df, *, manual=False):
    """Exact PMID, normalized CTD action, chemicalID, and gene symbol/ID keys."""
    x = df.copy()
    if manual:
        x = x.rename(columns={'InteractionActions':'relationship', 'ChemicalID':'chemicalID',
                              'GeneSymbol':'geneSymbol', 'GeneID':'geneID'})
    x['pmid'] = pd.to_numeric(x['pmid'], errors='coerce').astype('Int64').astype('string')
    x['chemicalID'] = x['chemicalID'].fillna('').astype(str).str.strip().str.upper()
    x['geneSymbol'] = x['geneSymbol'].fillna('').astype(str).str.strip().str.upper()
    if not manual:
        x['geneSymbol'] = x['geneSymbol'].str.replace(r'\..+$', '', regex=True)
    x['geneID'] = pd.to_numeric(x['geneID'], errors='coerce').astype('Int64').astype('string')
    x['gene_key'] = x['geneSymbol'].mask(x['geneSymbol'].isin(['','NAN','NA','<NA>']), x['geneID'])
    x['relationship_norm'] = x['relationship'].apply(normalize_relationship)
    return x

MERGE_KEYS = ['pmid','chemicalID','gene_key','relationship_norm']
manual_keys = matching_keys(ctd_ixns, manual=True)
# As in the combined notebook: one CTD reference per PMID/chemical/relationship/gene.
manual_keys = manual_keys.drop_duplicates(subset=['pmid','chemicalID','relationship','geneSymbol'])
ctd_reference_count = len(manual_keys)

def score_predictions(predictions):
    pred_keys = matching_keys(predictions)
    # Enforce one-to-one matching; repeated outputs should not create extra true positives.
    ref_keys = manual_keys[MERGE_KEYS].drop_duplicates()
    pred_unique = pred_keys[MERGE_KEYS].drop_duplicates()
    tp = len(pred_unique.merge(ref_keys, on=MERGE_KEYS, how='inner'))
    n = len(predictions)
    return {'predictions': n, 'matched': tp, 'ctd_reference': ctd_reference_count,
            'precision': tp / n if n else 0.0,
            'recall': tp / ctd_reference_count if ctd_reference_count else 0.0}


## Results: standardized vs. filtered

In [ ]:
# Run the same normalization order as in the combined notebook.
updated = update_dataframe(baseline)
updated = normalize_gene_symbols(
    updated, default_species='hsa', strict_conflicts=True,
    use_mirgenedb=True, use_hgnc=True, use_uniprot=True
)
updated = update_gene_symbol_(updated, ctd_genes, ctd_lookup=prepare_ctd(ctd_genes))

# Standardized-only: no quality filters, same dedup subset as combined notebook.
standardized = updated.drop_duplicates(
    subset=['pmid','chemicalID','geneSymbol','relationship'], keep='first'
).copy()

# Filtered: combined normalized-key dedup, then only sectionless-applicable filters.
filtered = filter_baseline(dedup_baseline(updated), ctd_chemicals, ctd_genes)

summary = pd.DataFrame([
    {'stage': 'standardized', **score_predictions(standardized)},
    {'stage': 'filtered', **score_predictions(filtered)},
])
summary.to_csv(SAVE_DIR / 'baseline_precision_recall.csv', index=False)
display(summary)
print('Saved:', SAVE_DIR / 'baseline_precision_recall.csv')
